# FaithfulnessEvaluator usage

## 1. What this metric measures

Faithfulness asks whether distinct factual claims in generated `output` are supported by authoritative `context` (`output → context`). It requires `context + output`. Optional `input` describes the generation task and remains useful for tracing, persistence, and debugging, but is not sent to the Faithfulness judge.

## 2. Imports

In [ ]:
from idp_eval import (
    EvaluationCase,
    EvaluationFramework,
    FaithfulnessEvaluator,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## 3. Judge configuration

Applications should inject configuration from their own settings/secrets layer. These are placeholders only. `create_gateway_judge(config=...)` works equivalently; evaluators are backend-independent.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)
framework = EvaluationFramework(
    evaluators=[FaithfulnessEvaluator(judge, verbose=True)],
)

## 4. Basic single-output example

Plain strings and structured `dict` / `list` / nested values are accepted. `render_value()` converts structured context and output into readable judge text. This example intentionally contains one supported and one unsupported claim.

In [ ]:
policy = {
    "cancellation_window_hours": 24,
    "refund_timeline": "5 business days",
}
case = EvaluationCase(
    case_id="faithfulness-basic-001",
    input="Answer the customer using the supplied cancellation policy.",
    context=policy,
    output="""
    Customers may cancel within 24 hours.
    Refunds are issued instantly.
    """,
)
result = framework.evaluate(case)["faithfulness"]

## 5. Understanding the result

The judge identifies factual claims and classifies each as `supported` (`1.0`) or `unsupported` (`0.0`). Python calculates `supported claims / total distinct factual claims`. Omitting information from context does not lower Faithfulness; Coverage handles omissions.

In [ ]:
{
    "score": result.score,
    "label": result.label,
    "explanation": result.explanation,
    "details": result.details,
}
result.details["claims"]

## 6. Multiple outputs and `evaluation_scope`

For separate generated artifacts, `individual` is often the most useful view because each artifact receives its own claim audit. `both` returns a collective judgment and each individual judgment in one call to `framework.evaluate()`. These views answer different questions and their scores should not be treated as equivalent. With `case_id='example-001'`, combined uses that ID and items use `example-001:0`, `example-001:1`, and `example-001:2`.

In [ ]:
answers = [
    "Customers may cancel within 24 hours.",
    "Refunds arrive within 5 business days.",
    "Refunds are issued instantly.",
]
individual_case = EvaluationCase(
    case_id="example-001", input=case.input, context=policy,
    output=answers, evaluation_scope="individual",
)
individual_results = framework.evaluate(individual_case)
individual_results["combined"]  # None
individual_results["individual"][0]["faithfulness"]

both_case = EvaluationCase(
    case_id="example-001", input=case.input, context=policy,
    output=answers, evaluation_scope="both",
)
both_results = framework.evaluate(both_case)
both_results["combined"]["faithfulness"]
both_results["individual"][0]["faithfulness"]

Return shapes: combined returns `{"faithfulness": EvaluationResult(...)}`; individual returns `{"combined": None, "individual": [{"faithfulness": ...}, ...]}`; both returns the combined metric mapping plus the individual list.

## 7. Optional async usage

Jupyter supports top-level `await`. The framework controls one shared judge-call concurrency limit.

In [ ]:
async_result = await framework.a_evaluate(case, max_concurrency=4)
async_result["faithfulness"]

## 8. `evaluate_many()`

`evaluate_many()` evaluates unrelated generation requests in order. Each case keeps its own scope and return shape.

In [ ]:
case_a = EvaluationCase(input="Answer one policy question.", context=policy, output=answers[0])
case_b = EvaluationCase(input="Review separate policy answers.", context=policy, output=answers, evaluation_scope="individual")
cases = [case_a, case_b]
many_results = framework.evaluate_many(cases)
async_many_results = await framework.a_evaluate_many(cases, max_concurrency=4)

## 9. Close resources

In [ ]:
judge.close()